## 2. Does a smaller catch prevent collapse?

**Model A takes a fixed amount each year. Model B takes a fraction of the population.**
For example, harvesting 10% per year removes 10 tonnes from a 100-tonne population,
but only 2 tonnes from a 20-tonne population.

We answer three questions:
1. Which population levels can remain steady under Model B?
2. How can we compare the two harvesting rules fairly?
3. Does reducing harvesting after collapse allow recovery?

These are **synthetic examples**, not observations of a real fish stock.

### 2.1 Steady populations under Model B

Use the same scaled variables as Task 1: $x=N/K$ is population relative to carrying
capacity, $a=A/K$ is the Allee threshold, $\tau=rt$ is scaled time, and $e=h/r$ is harvesting effort.

$$
\frac{dx}{d\tau}=x\left[(1-x)(x/a-1)-e\right]
$$

A steady population makes the right-hand side zero. One solution is $x=0$.
For the other solutions, divide by $x$ and rearrange:

$$
x^2-(1+a)x+a(1+e)=0
$$

The two roots are

$$
x_{\pm}=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}{2}
$$

They meet at the critical effort

$$
e_c=\frac{(1-a)^2}{4a}
$$

- **Below $e_c$:** the lower root $x_-$ is the recovery threshold; the upper root $x_+$ is stable.
  Starting below $x_-$ leads towards zero; starting above it leads towards $x_+$.
- **At $e_c$:** the roots meet. The steady state attracts from above but not from below
  (called *semistable*).
- **Above $e_c$:** no positive steady population remains.

Why? Factoring gives $f_B=-x(x-x_-)(x-x_+)/a$, whose signs are negative, positive,
negative across the three positive-population intervals. Also $f'_B(0)=-(1+e)<0$,
so zero stays stable even when harvesting stops.

**Example:** with $a=0.2$ and $e=0.4$, the recovery threshold is about $0.317K$
and the stable population is about $0.883K$. Here $e_c=0.8$.
The code below runs the shared experiment and shows these branches.

In [ ]:
import csv
import json
from IPython.display import Image, Markdown, display
from mm_allee.wp02_experiment import run_experiment

# Use ROOT from the notebook setup; run this experiment once.
config = json.loads((ROOT / "configs/wp02.json").read_text())
output = ROOT / "results/wp02-notebook"
manifest = run_experiment(config, output, ROOT)
display(Image(filename=str(output / "wp02_b_branches.png")))

### 2.2 Compare harvesting fairly

A fixed catch $H$ (tonnes/year) and an effort $h$ (per year) have different units.
We therefore use **two comparisons**:

1. **Same starting catch:** choose $H=hN_0$. Both rules remove the same amount initially;
   Model B's catch then changes with population.
2. **Same fraction of the limit:** use half of each model's own critical harvesting level.
   This compares equal distance from the collapse limit, but the initial catches differ.

For this example, $r=0.5$/year, $K=1000$ tonnes, $A=200$ tonnes, and the starting
population is 900 tonnes. We follow it for 60 years. The logistic model from Task 1
is included as a baseline.

**Mean catch = total catch / 60 years.** Total catch is the area under the catch-rate
curve: $\int_0^{60}hN(t)\,dt$ for B. For a fixed catch, harvesting stops at extinction.
The mean includes the years after shutdown.

The next table shows the important comparison results; the full CSV remains available.

In [ ]:
with (output / "wp02_comparison.csv").open() as f:
    rows = list(csv.DictReader(f))
labels = {"initial_catch": "Same starting catch", "threshold_fraction": "Half of each limit"}
outcomes = {"persistence": "Persists", "finite_extinction": "Collapses"}
lines = [
    "| Comparison | Model | Starting catch (t/year) | Mean catch (t/year) | Outcome |",
    "| --- | --- | --- | --- | --- |",
]
for row in rows:
    if float(row["initial_population"]) == 900 and float(row["fraction"]) == 0.5:
        lines.append(
            f"| {labels[row['protocol']]} | {row['model']} | "
            f"{float(row['initial_catch']):.2f} | {float(row['mean_catch']):.2f} | "
            f"{outcomes.get(row['theoretical_outcome'], row['theoretical_outcome'])} |"
        )
display(Markdown("\n".join(lines)))

**What this shows:** at the same starting catch (131.28 tonnes/year), A and B persist,
but the logistic population collapses because its maximum fixed catch is only
125 tonnes/year. At half of each model's limit, all three persist.

This does **not** prove that Allee growth is safer: the models have different growth
curves, and the conclusion depends on how the comparison is set up.

### 2.3 Does the population recover when harvesting falls?

**Method:** start at $N=K$, increase harvesting in steps to 1.3 times its critical
level, then decrease it to zero. Carry the population forward at every step—never
restart it at the turning point. Repeat with two dwell times (1 and 3 scaled time
units per ordinary step).

After harvesting returns to zero, compare:
- **No intervention:** let the population evolve on its own.
- **Restocking:** add $0.6K=600$ tonnes, shown separately by the dashed lines.

For A, reaching zero triggers harvest shutdown. For B, a positive population
approaches zero without reaching it in finite time; $x\leq10^{-6}$ is only a
smallness marker. The horizontal axis below is $\tau=rt$; here one unit equals two years.

In [ ]:
display(Image(filename=str(output / "wp02_history.png")))

### 2.4 Main conclusion

**Reducing harvesting does not guarantee recovery.** In both tested schedules,
the unassisted populations remain near zero. Once below the Allee threshold,
natural growth is negative even without harvesting. Adding 600 tonnes crosses
the threshold and allows recovery towards $K$.

This is **history dependence with a recovery barrier**, not a closed loop that
recovers by itself. The slower schedule also means longer exposure to harvesting,
so the experiment does not isolate the effect of speed alone.

**Checks and limits:** the experiment repeats the histories with tighter solver
settings; population differences are below $2\times10^{-9}$ in the recorded run.
Very small B populations have no claimed relative accuracy. Outcomes near an
uncertain starting-population boundary are labelled unresolved. The code, full
comparison table and run settings are saved by the experiment; no real-stock
prediction is claimed.

**Remember:** a smaller catch at low population helps reduce removal, but it does
not remove the Allee recovery threshold.